# 02 · Disperse a star on one SCA

In [notebook 01](01_spectra_to_counts.ipynb) we turned a source into a **count-rate spectrum**. Now we disperse it: place a star at a fixed pixel on one detector and run it through the **optical model** (where does each wavelength land?) and the **PSF** (what does it look like there?) to produce a simulated first-order spectrum on the SCA.

You will meet the three objects the disperser is built from —

- the **optical-model payload** (`optical_model_jax.make_sca_payload`),
- the **PSF payload** (`psf_model.get_or_make_psf_payload`),
- the **star disperser** (`star_disperser.make_star_disperser`),

— learn why the **first call is slow** (JAX compilation) and how to keep that cost down, disperse **all three spectral orders** separately and co-added, and finally **save a grism FITS** and read it back. This is the first notebook that actually computes a dispersed image; on a laptop CPU it still runs in well under a minute at the resolution we use here.

> **Needs:** the `psf` asset hydrated for the SCA we use (`pixi run hydrate --only psf --sca 5`, or the full `pixi run hydrate`). See `docs/SETUP.md`.

## 0 · Setup

In [ ]:
import os
from pathlib import Path
# Cache compiled JAX kernels on disk so re-running this notebook (or later ones)
# skips recompilation. Must be set BEFORE jax is imported. See §5.
os.environ.setdefault("JAX_COMPILATION_CACHE_DIR",
                      str(Path.home() / ".cache" / "roman_grs_jax"))

import time
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import AsinhNorm

import jax
import jax.numpy as jnp

from roman_disperser import paths, psf_model, star_disperser, pipeline
from roman_disperser.elements import GRISM
from roman_disperser.optical_model import RomanOpticalModel
import roman_disperser.optical_model_jax as omj
import tutorial_helpers as th

print("JAX backend:", jax.default_backend(), "| devices:", jax.devices())
print("reference data:", paths.data_dir())

SCA = 5            # detector (WFI05); we hydrated its PSF cache
DETECTOR = f"WFI{SCA:02d}"
element = GRISM    # the dispersing element (notebook 09 swaps in PRISM)

## 1 · The optical-model payload

The Roman grism optical model is a calibrated, per-detector description of where light of a given wavelength lands and how the spectral trace curves. The full model is a NumPy/dataclass object, `RomanOpticalModel`; for fast, JIT-able use we extract a lightweight **payload** for one (SCA, order) with `optical_model_jax.make_sca_payload`. (Orders are *strings*: `"0"`, `"1"`, `"2"`.)

We'll start with the science order, **order 1**.

In [ ]:
model = RomanOpticalModel(config_file=str(paths.optical_model_path(element=element)))
opt1 = omj.make_sca_payload(model, sca=SCA, order="1")
print("optical payload keys:", list(opt1.keys()))

## 2 · The PSF payload

`psf_model` provides the wavelength- and field-dependent PSF as a precomputed, cached grid for an (SCA, order); `get_or_make_psf_payload` simply loads it (generating from STPSF is slow, so we hydrate the cache ahead of time).

In [ ]:
psf1 = psf_model.get_or_make_psf_payload(
    detector=DETECTOR, order="1", element=element,
    cache_dir=str(paths.psf_cache_dir()), verbose=False,
)
print("PSF grid shape:", psf1["psf_grid"].shape,
      "(field_y, field_x, n_wl, ny, nx)")
print("oversample:", psf1["oversample"])

## 3 · The star disperser

`star_disperser.make_star_disperser(psf_payload, optical_payload)` returns a **callable** specialised to that (SCA, order). Its signature is

```python
disperse(xsca, ysca, wavelengths_um, counts, output) -> output
```

- `xsca, ysca` — source position in **1-indexed FITS pixels** (pixel *n* is centred at *n.0*); the detector is 4088 × 4088;
- `wavelengths_um` — wavelength grid in **microns**;
- `counts` — the count-rate spectrum on that grid (from notebook 01);
- `output` — a `(4088, 4088)` array the dispersed flux is **added into**, so you can accumulate many sources (and many orders) into one image.

The non-array bits of the payloads (strings, dict structure) are captured in a closure and the numerical core is `jax.jit`-compiled.

In [ ]:
disperse1 = star_disperser.make_star_disperser(psf1, opt1)

## 4 · A count-rate spectrum and a position

We reuse `template_to_counts` from notebook 01 — a **G0V star, AB = 16** — on the production **2 Å** grid (≈5500 points). The grid spacing matters for accuracy: the trace is sampled wavelength by wavelength, so coarsening the grid noticeably degrades the dispersed spectrum. We use the science grid throughout and place the star near the detector centre.

In [ ]:
wl_um, _, _ = th.wavelength_grid(element)                 # 2 Å production grid
_, counts = th.template_to_counts("g0v", mag=16.0, sca=SCA, order="1", wl_um=wl_um)
wl_j = jnp.asarray(wl_um)
counts1_j = jnp.asarray(counts)
print(f"{wl_j.size} wavelength samples; total order-1 rate {counts.sum():.0f} e-/s")

X_STAR, Y_STAR = 2000.0, 2000.0      # fixed pixel position (1-indexed)

## 5 · JAX warm-up — why the first call is slow

JAX **traces and compiles** a function the first time it sees a given set of array shapes/dtypes, then runs the fast compiled version on every later call with matching shapes. So the *first* dispersion includes a one-time compile cost (seconds); subsequent calls are much faster. JAX is also **asynchronous** — we call `.block_until_ready()` to force the computation to finish before we read the clock.

This "warm-up" pattern (do one throwaway call, then time the real work) is why the production pipeline compiles once up front. We also set `JAX_COMPILATION_CACHE_DIR` in cell 0: compiled kernels are written to disk, so *re-running this notebook in a fresh session* skips compilation entirely.

In [ ]:
output = jnp.zeros((4088, 4088), dtype=jnp.float32)

t = time.time()
img1 = disperse1(X_STAR, Y_STAR, wl_j, counts1_j, output)
img1.block_until_ready()
print(f"first call  (compile + run): {time.time()-t:5.2f} s")

t = time.time()
img1 = disperse1(X_STAR, Y_STAR, wl_j, counts1_j, output)
img1.block_until_ready()
print(f"second call (cached)       : {time.time()-t:5.2f} s")

img1 = np.asarray(img1)
print(f"dispersed total: {img1.sum():.0f} e-/s")

In [ ]:
def trace_extent(img, pad=10):
    # bounding box of the nonzero (dispersed) pixels, with padding
    ys, xs = np.nonzero(img)
    return (max(xs.min()-pad, 0), min(xs.max()+pad, img.shape[1]),
            max(ys.min()-pad, 0), min(ys.max()+pad, img.shape[0]))

x0, x1, y0, y1 = trace_extent(img1)
vmax = img1.max()
fig, (axf, axz) = plt.subplots(1, 2, figsize=(10, 5))
# hard asinh stretch so the faint trace stands out at full scale
axf.imshow(img1, origin="lower", cmap="inferno",
           norm=AsinhNorm(linear_width=vmax*0.001, vmin=0, vmax=vmax))
axf.plot(X_STAR-1, Y_STAR-1, "c+", ms=10)
axf.set(title="order 1 — full SCA", xlabel="x [pix]", ylabel="y [pix]")
axz.imshow(img1[y0:y1, x0:x1], origin="lower", cmap="inferno",
           norm=AsinhNorm(linear_width=vmax*0.01, vmin=0, vmax=vmax))
axz.set(title="order 1 — zoom on the trace", xticks=[], yticks=[])
fig.tight_layout()

The first-order spectrum is the bright streak; dispersion runs along **y**. The cyan `+` marks the (undispersed) source position — note the spectrum is offset from it, which is exactly the geometry the extraction notebook (04) inverts.

## 6 · Flux accounting — does the PSF catch every photon?

Before adding the other orders, a sanity check: did the order-1 spectrum conserve flux? Compare the integrated **input** count rate with what actually landed on the detector.

In [ ]:
deposited = img1.sum()
print(f"input integrated counts : {counts.sum():8.1f} e-/s")
print(f"deposited on detector   : {deposited:8.1f} e-/s")
print(f"missing                 : {100*(1 - deposited/counts.sum()):.1f} %")

# Cause: the cached PSF stamps integrate to < 1 (their encircled energy within the
# 5" stamp), and it depends on wavelength because the PSF broadens toward the red.
ee_grid = np.asarray(psf1["psf_grid"]).sum(axis=(-2, -1))   # (field_y, field_x, n_wl)
ee_wl = ee_grid.mean(axis=(0, 1))
wlg = np.asarray(psf1["wl_grid"])

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(wlg, ee_wl, "o-", ms=3, color="C0")
ax.set(xlabel="wavelength [µm]", ylabel="PSF encircled energy",
       title="fraction of a source's light inside the 5″ PSF stamp")
fig.tight_layout()
print(f"PSF encircled energy: {ee_wl[0]:.3f} (blue) -> {ee_wl[-1]:.3f} (red)")

# The deficit is *exactly* the spectrum-weighted PSF EE at the source position:
from scipy.interpolate import RegularGridInterpolator
sx, sy = np.asarray(psf1["spatial_x"]), np.asarray(psf1["spatial_y"])
ee_src = RegularGridInterpolator((sy, sx, wlg), ee_grid)(
    np.column_stack([np.full_like(wl_um, Y_STAR), np.full_like(wl_um, X_STAR), wl_um]))
print(f"predicted Σ counts·EE(source,λ) = {np.sum(counts*ee_src):.1f}  (matches deposited {deposited:.1f})")

So ~3% of the order-1 light never reaches the detector — it sits in the PSF wings beyond the 5″ stamp. Because the PSF broadens with wavelength, the loss runs from ~2% in the blue to ~4% in the red: a ~2% blue→red tilt on the spectrum. The deposit is **exactly** Σ counts(λ)·EE(source, λ), so the disperser conserves flux *given* the stamps it is handed; the deficit is a property of the (truncated) PSF cache.

Does it matter? Per pixel, hardly — the missing wings are very faint. For this AB = 16 star the cross-dispersion surface brightness drops below the Roman grism **zodiacal background** (~0.65 e⁻/s/pix) by r ≈ 0.4″, and at the 2.5″ stamp edge it is ~140× below the sky — unrecoverable for any source fainter than F158 ≈ 11. But the ~3% throughput deficit and the ~2% spectral tilt are real for absolute spectrophotometry, and should be reconciled with how the sensitivity curves define encircled energy.

## 7 · All three orders — separately and co-added

A grism splits light into multiple orders. Order 1 is the science order; order 0 is the (nearly undispersed) direct image; order 2 is a faint, more-dispersed copy. We build a disperser per order, give each the count-rate spectrum computed with **that order's** sensitivity curve (the curves already encode each order's efficiency, so no ad-hoc fudge factor is needed), and disperse.

In [ ]:
orders = list(element.orders)                       # ("0", "1", "2") for the grism
opt = {o: omj.make_sca_payload(model, sca=SCA, order=o) for o in orders}

# element= maps each order to its STPSF filter — including the fact that
# order 2 shares the order-1 PSF (element.stpsf_filters["2"] == "GRISM1").
psf = {o: psf_model.get_or_make_psf_payload(detector=DETECTOR, order=o, element=element,
            cache_dir=str(paths.psf_cache_dir()), verbose=False)
       for o in orders}

disp = {o: star_disperser.make_star_disperser(psf[o], opt[o]) for o in orders}

counts_o = {}
for o in orders:
    _, c = th.template_to_counts("g0v", mag=16.0, sca=SCA, order=o, wl_um=wl_um)
    counts_o[o] = jnp.asarray(c)

per_order = {}
combined = np.zeros((4088, 4088), dtype=np.float32)
for o in orders:
    out = disp[o](X_STAR, Y_STAR, wl_j, counts_o[o],
                  jnp.zeros((4088, 4088), jnp.float32))
    out.block_until_ready()
    per_order[o] = np.asarray(out)
    combined += per_order[o]
    print(f"order {o}: total {per_order[o].sum():8.1f} e-/s")
print(f"combined: {combined.sum():.1f} e-/s")

In [ ]:
# All three orders for one star, in a single view. The background is exactly
# zero, so we can push the asinh stretch very hard (tiny linear_width) to bring
# the faint 0th and 2nd orders up next to the bright 1st order. The white circle
# marks the source's direct-image position.
ys, xs = np.nonzero(combined)
pad = 60
bx0, bx1 = xs.min()-pad, xs.max()+pad
by0, by1 = ys.min()-pad, ys.max()+pad
vmax = combined.max()

fig, ax = plt.subplots(figsize=(5, 8))
ax.imshow(combined[by0:by1, bx0:bx1], origin="lower", cmap="inferno",
          extent=[bx0, bx1, by0, by1],
          norm=AsinhNorm(linear_width=vmax*5e-5, vmin=0, vmax=vmax))
for o in orders:
    yy, xx = np.nonzero(per_order[o])
    ax.text(xx.mean()+30, yy.mean(), f"order {o}", color="cyan", fontsize=9, va="center")
ax.plot(X_STAR, Y_STAR, "o", mfc="none", mec="white", ms=10)
ax.text(X_STAR+30, Y_STAR, "source", color="white", fontsize=9, va="center")
ax.set(title="all three orders (hard asinh stretch)", xlabel="x [pix]", ylabel="y [pix]")
fig.tight_layout()

Order 1 dominates; orders 0 and 2 are faint (the sensitivity curves put ~0.3 % of the order-1 throughput into each). Real exposures contain all orders at once, which is why **order-0 light from one source can contaminate the order-1 spectrum of another** — a hazard we return to in notebook 04.

## 8 · Save a grism FITS, then read it back

We store the result in the same FITS layout the production pipeline writes, using `pipeline.write_fits`, so downstream tools (the extractor in notebook 04) read our hand-built image exactly as they would a pipeline image:

- **PrimaryHDU** — pointing/metadata headers;
- **MODEL** — the noiseless count-*rate* image (e⁻/s);
- **ISIM** — a Poisson realisation in counts (rate × exposure time), ready for noise/instrument modelling.

In notebook mode we placed the star by pixel, so the pointing keywords are **nominal metadata** for downstream tools; notebooks 03 and 07 show how pixel positions actually come from a sky pointing (RA/Dec/PA).

In [ ]:
EXPTIME, SEED = 190.22, 0
key = jax.random.PRNGKey(SEED)
isim = np.asarray(jax.random.poisson(key, jnp.asarray(combined) * EXPTIME)
                  ).astype(np.float32)

out_path = Path("outputs"); out_path.mkdir(exist_ok=True)
fits_file = out_path / f"grism_demo_detSCA{SCA:02d}.fits"
pipeline.write_fits(
    combined, isim, str(fits_file),
    pointing_ra=10.0, pointing_dec=0.0, pointing_pa=0.0,
    sca=SCA, exptime=EXPTIME, rng_key_data=np.asarray(key), seed=SEED,
    extra_headers={"MA_TABLE": (1036, "MA table number"),
                   "OPTELEM": (element.name, "Dispersing element")},   # as production writes it
)
print("wrote", fits_file)

In [ ]:
from astropy.io import fits

with fits.open(fits_file) as hdul:
    print("HDUs:", [h.name for h in hdul])
    print(f"pointing: RA={hdul[0].header['WFICENRA']} "
          f"Dec={hdul[0].header['WFICENDEC']} PA={hdul[0].header['WFICENPA']} "
          f"SCA={hdul[0].header['DETNUM']}")
    model_img = hdul["MODEL"].data
    isim_img = hdul["ISIM"].data

x0, x1, y0, y1 = trace_extent(model_img, pad=15)
fig, (a0, a1) = plt.subplots(1, 2, figsize=(9, 6))
for ax, img, title in [(a0, model_img, "MODEL (count rate, e⁻/s)"),
                       (a1, isim_img, "ISIM (Poisson counts)")]:
    ax.imshow(img[y0:y1, x0:x1], origin="lower", cmap="inferno",
              norm=AsinhNorm(linear_width=img.max()*0.01, vmin=0, vmax=img.max()))
    ax.set(title=title, xticks=[], yticks=[])
fig.tight_layout()

## Recap

- A disperser is built from three pieces — `make_sca_payload` (optics), `get_or_make_psf_payload` (PSF), `make_star_disperser` (the callable) — and accumulates flux into a `(4088, 4088)` image.
- The **first call compiles** (slow); warm up once and cache to disk with `JAX_COMPILATION_CACHE_DIR`.
- Build one disperser **per order** and co-add; per-order **sensitivity curves** carry the efficiencies.
- `pipeline.write_fits` gives a **pipeline-compatible** MODEL/ISIM FITS that the rest of the series can consume.

**Next — [03 · Stars and galaxies, and a roll](03_stars_galaxies_roll.ipynb).** We add extended (Sérsic) galaxies via `galaxy_disperser`, build a small mixed field, and re-observe the same field at a different **roll angle** to see the spectra rotate — the setup for the contamination story in notebook 06.